# Kisan Mitra — Rice Disease Classifier v3

Same Kaggle **Paddy Doctor** data as v2 (~10,400 labelled images, 10 classes), trained properly.

| | v2 | v3 |
|---|---|---|
| Backbone | MobileNetV2 | EfficientNetV2-S (or ConvNeXt-Tiny) |
| Fine-tuning | last 20 layers, LR 1e-5 | whole network (BatchNorm frozen), AdamW, warmup + cosine |
| Input | 224×224 (squashed) | 384×288 portrait (keeps the photo's shape) |
| Augmentation | flip / rotate / zoom / contrast | random crop, flips, rotation, colour, blur, JPEG, MixUp / CutMix |
| Loss | plain cross-entropy | label smoothing 0.1 + softened class weights |
| Models | 1 | 5-fold ensemble + flip test-time augmentation |
| Extra data | — | pseudo-labelled Kaggle test images |
| Confidence | raw softmax | temperature-calibrated, threshold picked from out-of-fold data |

**Runtime:** use a GPU (Runtime → Change runtime type). A100 / L4 is best. A T4 works but the full
5-fold run with pseudo-labelling takes several hours, so set `FOLDS_TO_TRAIN = [0]` for a first run.
Progress is saved to Google Drive after every fold; if Colab disconnects, re-run all cells and
finished folds are skipped.

**Output:** `paddy_disease_model_fold*.tflite` + `model_meta.json`. Copy them into
`backend/models/`; the backend loads them as an ensemble automatically.

**Honest accuracy:** the numbers printed here are on Paddy Doctor photos. Real farmer photos will
score lower. Collect 100–200 labelled photos from the field and test on them before trusting the model.

## 1. Setup

In [ ]:
!pip install -q kagglehub

In [ ]:
import gc
import json
import os
import shutil
from concurrent.futures import ThreadPoolExecutor
from datetime import date

import keras
import numpy as np
import pandas as pd
import tensorflow as tf
from keras import layers
from PIL import Image, ImageOps
from scipy.optimize import minimize_scalar
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.model_selection import StratifiedKFold

print("TensorFlow", tf.__version__, "| Keras", keras.__version__)
GPUS = tf.config.list_physical_devices("GPU")
print("GPU:", GPUS or "none — training will be very slow")

## 2. Configuration

Everything you might want to change is here.

In [ ]:
BACKBONE = "efficientnetv2s"      # "efficientnetv2s" | "efficientnetv2b3" | "convnext_tiny"
IMAGENET_WEIGHTS = "imagenet"

# Paddy Doctor photos are 480 wide x 640 tall. Train at the same 3:4 portrait shape.
IMG_H, IMG_W = 384, 288
CACHE_H, CACHE_W = 512, 384       # images are pre-resized once to this size (faster loading)
BATCH_SIZE = 32

N_FOLDS = 5
FOLDS_TO_TRAIN = [0, 1, 2, 3, 4]  # [0] = one model, ~5x faster, about 1% less accurate

HEAD_EPOCHS = 2                   # train only the new classifier head first
EPOCHS = 20                       # then fine-tune the whole network
PEAK_LR = 3e-4
WEIGHT_DECAY = 0.05
LABEL_SMOOTHING = 0.1
CLASS_WEIGHT_POWER = 0.5          # 0 = no class weights, 1 = fully balanced
MIX_PROB = 0.5                    # chance per batch of MixUp or CutMix

PSEUDO_LABEL = True               # second round using confident predictions on Kaggle's test images
PSEUDO_THRESHOLD = 0.95
PL_EPOCHS = 6
PL_PEAK_LR = 1e-4

TARGET_ACCEPTED_ACCURACY = 0.97   # confidence threshold is chosen to reach this accuracy
QUANTIZE = "dynamic"              # "dynamic" (~4x smaller) | "float16" | "none"
TFLITE_CHECK_SAMPLES = 200        # val images per fold used to check the exported .tflite

MIXED_PRECISION = bool(GPUS)
SEED = 42

WORK_DIR = "/content/kisan_mitra_v3"                     # fast local disk
SAVE_DIR = "/content/drive/MyDrive/kisan_mitra/model_v3"  # persistent (Google Drive)

# Folder name in the dataset -> name used by the app (must match backend/diseases.json)
DISPLAY_NAMES = {
    "bacterial_leaf_blight":    "Bacterial Leaf Blight",
    "bacterial_leaf_streak":    "Bacterial Leaf Streak",
    "bacterial_panicle_blight": "Bacterial Panicle Blight",
    "blast":                    "Blast",
    "brown_spot":               "Brown Spot",
    "dead_heart":               "Dead Heart",
    "downy_mildew":             "Downy Mildew",
    "hispa":                    "Hispa",
    "normal":                   "Normal",
    "tungro":                   "Tungro",
}

In [ ]:
try:
    from google.colab import drive
    drive.mount("/content/drive")
except ImportError:
    SAVE_DIR = os.path.join(WORK_DIR, "saved")
    print("Not in Colab — saving to", SAVE_DIR)

In [ ]:
os.makedirs(WORK_DIR, exist_ok=True)
os.makedirs(SAVE_DIR, exist_ok=True)
keras.utils.set_random_seed(SEED)
keras.mixed_precision.set_dtype_policy("mixed_float16" if MIXED_PRECISION else "float32")
print("Mixed precision:", MIXED_PRECISION)

## 3. Download the dataset

You need a Kaggle API token and must have accepted the
[competition rules](https://www.kaggle.com/competitions/paddy-disease-classification/rules).

In [ ]:
import getpass
token = getpass.getpass("Paste your Kaggle API token: ")
os.makedirs(os.path.expanduser("~/.kaggle"), exist_ok=True)
with open(os.path.expanduser("~/.kaggle/access_token"), "w") as f:
    f.write(token.strip())
os.chmod(os.path.expanduser("~/.kaggle/access_token"), 0o600)

import kagglehub
DATA_DIR = kagglehub.competition_download("paddy-disease-classification")
print("Data at:", DATA_DIR, os.listdir(DATA_DIR))

## 4. Data table, folds and image cache

Images are split into `N_FOLDS` stratified folds; each model is validated on the fold it never saw.

Every image is resized once to 512×384 with the same steps the backend uses (EXIF rotation,
landscape → portrait), so training sees images the way the app will.

In [ ]:
CLASS_FOLDERS = sorted(DISPLAY_NAMES)          # alphabetical, same order as v2
CLASS_NAMES = [DISPLAY_NAMES[c] for c in CLASS_FOLDERS]
NUM_CLASSES = len(CLASS_FOLDERS)

df = pd.read_csv(os.path.join(DATA_DIR, "train.csv"))
assert df.image_id.is_unique   # the image cache is keyed by file name
assert set(df.label) == set(CLASS_FOLDERS), f"Unexpected labels: {set(df.label) ^ set(CLASS_FOLDERS)}"
df["src"] = [os.path.join(DATA_DIR, "train_images", l, i) for l, i in zip(df.label, df.image_id)]
df["y"] = df.label.map({c: i for i, c in enumerate(CLASS_FOLDERS)})

df["fold"] = -1
skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
for fold, (_, val_idx) in enumerate(skf.split(df, df.y)):
    df.loc[val_idx, "fold"] = fold

test_dir = os.path.join(DATA_DIR, "test_images")
test_df = pd.DataFrame({"image_id": sorted(os.listdir(test_dir))})
test_df["src"] = [os.path.join(test_dir, i) for i in test_df.image_id]

counts = df.label.value_counts().reindex(CLASS_FOLDERS)
weights = (len(df) / (NUM_CLASSES * counts)) ** CLASS_WEIGHT_POWER
CLASS_WEIGHTS = (weights / weights.mean()).to_numpy(dtype="float32")
df["w"] = CLASS_WEIGHTS[df.y]

print(f"{len(df)} labelled images, {len(test_df)} unlabelled test images")
print(pd.DataFrame({"images": counts.values, "weight": CLASS_WEIGHTS.round(2)}, index=CLASS_NAMES))

In [ ]:
def prepare_image(img):
    """Same as backend/inference.py prepare_image(): EXIF rotation, landscape -> portrait."""
    img = ImageOps.exif_transpose(img).convert("RGB")
    if img.width > img.height:
        img = img.rotate(90, expand=True)
    return img


def cache_image(src, dst):
    if os.path.exists(dst):
        return
    with Image.open(src) as img:
        img = prepare_image(img).resize((CACHE_W, CACHE_H), Image.Resampling.BILINEAR)
    img.save(dst, quality=95)


def build_cache(frame, subdir):
    out_dir = os.path.join(WORK_DIR, f"cache_{CACHE_W}x{CACHE_H}", subdir)
    os.makedirs(out_dir, exist_ok=True)
    dsts = [os.path.join(out_dir, os.path.basename(s)) for s in frame.src]
    with ThreadPoolExecutor(16) as pool:
        list(pool.map(cache_image, frame.src, dsts))
    return dsts


df["path"] = build_cache(df, "train")   # image ids are unique across classes
test_df["path"] = build_cache(test_df, "test")
print("Cached", len(df) + len(test_df), "images")

## 5. Input pipeline and augmentation

Augmentations imitate farmer photos: different framing (random crop), orientation, lighting and
colour, slight blur, sensor noise, small patches of dirt or shadow, and heavy JPEG compression.
They are applied randomly every time an image is loaded, so each epoch sees different versions of
each photo. This beats saving augmented copies to disk, where copies of one photo can end up on both
sides of the train/validation split and inflate the accuracy. MixUp / CutMix blend pairs of images, which
reduces overfitting and over-confident predictions.

In [ ]:
AUTOTUNE = tf.data.AUTOTUNE
_rotate = layers.RandomRotation(0.05, fill_mode="reflect", seed=SEED, dtype="float32")  # up to ~18 degrees


def _augment(raw, label, weight):
    img = tf.io.decode_jpeg(raw, channels=3)
    img = tf.cond(tf.random.uniform([]) < 0.3,
                  lambda: tf.image.random_jpeg_quality(img, 40, 95),
                  lambda: img)
    img.set_shape([None, None, 3])

    # Random crop covering 50-100% of the image, then resize to the model input
    begin, size, _ = tf.image.sample_distorted_bounding_box(
        tf.shape(img), bounding_boxes=tf.constant([0.0, 0.0, 1.0, 1.0], shape=[1, 1, 4]),
        min_object_covered=0.0, aspect_ratio_range=(0.6, 0.95), area_range=(0.5, 1.0),
        max_attempts=10, use_image_if_no_bounding_boxes=True)
    img = tf.slice(img, begin, size)
    img = tf.image.resize(img, (IMG_H, IMG_W), antialias=True)

    img = tf.image.random_flip_left_right(img)
    img = tf.image.random_flip_up_down(img)

    img = img / 255.0
    img = tf.image.random_brightness(img, 0.15)
    img = tf.image.random_contrast(img, 0.8, 1.2)
    img = tf.image.random_saturation(img, 0.8, 1.2)
    img = tf.image.random_hue(img, 0.03)
    img = tf.clip_by_value(img, 0.0, 1.0) * 255.0

    # Slight blur, like an out-of-focus phone photo
    img = tf.cond(tf.random.uniform([]) < 0.2,
                  lambda: tf.image.resize(tf.image.resize(img, (IMG_H // 2, IMG_W // 2), antialias=True),
                                          (IMG_H, IMG_W)),
                  lambda: img)

    # Sensor grain / noise, like a cheap phone camera in low light
    img = tf.cond(tf.random.uniform([]) < 0.2,
                  lambda: img + tf.random.normal(tf.shape(img), stddev=tf.random.uniform([], 3.0, 12.0)),
                  lambda: img)

    # Small patch covering part of the leaf (dirt, shadow, a finger). Kept small (2-8% of the
    # image) so it rarely hides the whole lesion and changes what the label should be.
    img = tf.cond(tf.random.uniform([]) < 0.25, lambda: _occlude(img), lambda: img)

    img = tf.clip_by_value(img, 0.0, 255.0)
    return img, tf.one_hot(label, NUM_CLASSES), weight


def _occlude(img):
    area = tf.random.uniform([], 0.02, 0.08) * IMG_H * IMG_W
    aspect = tf.random.uniform([], 0.5, 2.0)
    h = tf.clip_by_value(tf.cast(tf.sqrt(area * aspect), tf.int32), 1, IMG_H)
    w = tf.clip_by_value(tf.cast(tf.sqrt(area / aspect), tf.int32), 1, IMG_W)
    y = tf.random.uniform([], 0, IMG_H - h + 1, tf.int32)
    x = tf.random.uniform([], 0, IMG_W - w + 1, tf.int32)
    rows = tf.range(IMG_H)[:, None]
    cols = tf.range(IMG_W)[None, :]
    mask = tf.cast((rows >= y) & (rows < y + h) & (cols >= x) & (cols < x + w), tf.float32)[:, :, None]
    # Dark, earthy colour (soil, mud, shadow), partly see-through
    colour = tf.random.uniform([], 20.0, 110.0) * tf.random.uniform([1, 1, 3], 0.7, 1.0)
    mask = mask * tf.random.uniform([], 0.5, 1.0)
    return img * (1 - mask) + colour * mask


def _beta(alpha):
    a = tf.random.gamma([], alpha)
    b = tf.random.gamma([], alpha)
    return a / (a + b)


def _mixup(images, labels, weights, idx):
    lam = _beta(0.4)
    return (lam * images + (1 - lam) * tf.gather(images, idx),
            lam * labels + (1 - lam) * tf.gather(labels, idx),
            lam * weights + (1 - lam) * tf.gather(weights, idx))


def _cutmix(images, labels, weights, idx):
    lam = _beta(1.0)
    cut_h = tf.cast(IMG_H * tf.sqrt(1 - lam), tf.int32)
    cut_w = tf.cast(IMG_W * tf.sqrt(1 - lam), tf.int32)
    cy = tf.random.uniform([], 0, IMG_H, tf.int32)
    cx = tf.random.uniform([], 0, IMG_W, tf.int32)
    y1, y2 = tf.clip_by_value(cy - cut_h // 2, 0, IMG_H), tf.clip_by_value(cy + cut_h // 2, 0, IMG_H)
    x1, x2 = tf.clip_by_value(cx - cut_w // 2, 0, IMG_W), tf.clip_by_value(cx + cut_w // 2, 0, IMG_W)
    rows = tf.range(IMG_H)[:, None]
    cols = tf.range(IMG_W)[None, :]
    box = (rows >= y1) & (rows < y2) & (cols >= x1) & (cols < x2)
    mask = tf.cast(box, images.dtype)[None, :, :, None]
    images = images * (1 - mask) + tf.gather(images, idx) * mask
    lam = 1 - tf.cast((y2 - y1) * (x2 - x1), tf.float32) / (IMG_H * IMG_W)   # actual pasted area
    return (images,
            lam * labels + (1 - lam) * tf.gather(labels, idx),
            lam * weights + (1 - lam) * tf.gather(weights, idx))


def _mix_batch(images, labels, weights):
    idx = tf.random.shuffle(tf.range(tf.shape(images)[0]))
    r = tf.random.uniform([])
    return tf.cond(r < MIX_PROB / 2,
                   lambda: _mixup(images, labels, weights, idx),
                   lambda: tf.cond(r < MIX_PROB,
                                   lambda: _cutmix(images, labels, weights, idx),
                                   lambda: (images, labels, weights)))


def make_train_ds(frame):
    ds = tf.data.Dataset.from_tensor_slices(
        (frame.path.values, frame.y.values.astype("int32"), frame.w.values.astype("float32")))
    ds = ds.map(lambda p, y, w: (tf.io.read_file(p), y, w), num_parallel_calls=AUTOTUNE).cache()
    ds = ds.shuffle(len(frame), seed=SEED, reshuffle_each_iteration=True)
    ds = ds.map(_augment, num_parallel_calls=AUTOTUNE).batch(BATCH_SIZE, drop_remainder=True)
    ds = ds.map(lambda x, y, w: (_rotate(x, training=True), y, w), num_parallel_calls=AUTOTUNE)
    return ds.map(_mix_batch, num_parallel_calls=AUTOTUNE).prefetch(AUTOTUNE)


def _load_eval(path):
    img = tf.io.decode_jpeg(tf.io.read_file(path), channels=3)
    return tf.image.resize(img, (IMG_H, IMG_W), antialias=True)


def make_eval_ds(frame, labels=False):
    ds = tf.data.Dataset.from_tensor_slices(frame.path.values).map(_load_eval, num_parallel_calls=AUTOTUNE)
    if labels:
        ys = tf.data.Dataset.from_tensor_slices(tf.one_hot(frame.y.values, NUM_CLASSES))
        ds = tf.data.Dataset.zip((ds, ys))
    return ds.batch(BATCH_SIZE).prefetch(AUTOTUNE)


# Quick look at one augmented batch
_x, _y, _w = next(iter(make_train_ds(df.head(BATCH_SIZE * 2))))
print("batch", _x.shape, _y.shape, _w.shape, "pixel range", float(tf.reduce_min(_x)), float(tf.reduce_max(_x)))

## 6. Model and training loop

In [ ]:
def build_model(weights=IMAGENET_WEIGHTS):
    shape = (IMG_H, IMG_W, 3)
    # All three backbones include their own preprocessing layer and take raw 0-255 pixels,
    # which is what the backend sends (MODEL_PREPROCESS=none).
    if BACKBONE == "efficientnetv2s":
        base = keras.applications.EfficientNetV2S(include_top=False, weights=weights, input_shape=shape)
    elif BACKBONE == "efficientnetv2b3":
        base = keras.applications.EfficientNetV2B3(include_top=False, weights=weights, input_shape=shape)
    elif BACKBONE == "efficientnetv2b0":
        base = keras.applications.EfficientNetV2B0(include_top=False, weights=weights, input_shape=shape)
    elif BACKBONE == "convnext_tiny":
        base = keras.applications.ConvNeXtTiny(include_top=False, weights=weights, input_shape=shape)
    else:
        raise ValueError(f"Unknown BACKBONE {BACKBONE!r}")

    inputs = keras.Input(shape)
    x = base(inputs, training=False)          # keeps BatchNorm in inference mode while fine-tuning
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dropout(0.3)(x)
    outputs = layers.Dense(NUM_CLASSES, activation="softmax", dtype="float32")(x)
    return keras.Model(inputs, outputs), base


def set_base_trainable(base, trainable):
    base.trainable = trainable
    if trainable:
        for layer in base.layers:
            if isinstance(layer, layers.BatchNormalization):
                layer.trainable = False


def compile_model(model, peak_lr, total_steps, warmup_steps):
    schedule = keras.optimizers.schedules.CosineDecay(
        initial_learning_rate=peak_lr * 0.01 if warmup_steps else peak_lr,
        decay_steps=max(total_steps - warmup_steps, 1),
        alpha=0.01,
        warmup_target=peak_lr if warmup_steps else None,
        warmup_steps=warmup_steps,
    )
    optimizer = keras.optimizers.AdamW(learning_rate=schedule, weight_decay=WEIGHT_DECAY, clipnorm=1.0)
    optimizer.exclude_from_weight_decay(var_names=["bias", "gamma", "beta"])
    model.compile(
        optimizer=optimizer,
        loss=keras.losses.CategoricalCrossentropy(label_smoothing=LABEL_SMOOTHING),
        metrics=["accuracy"],
    )


def weights_path(round_name, fold):
    return os.path.join(SAVE_DIR, f"{BACKBONE}_{round_name}_fold{fold}.weights.h5")


def train_fold(fold, round_name, extra=None, init_weights=None,
               head_epochs=HEAD_EPOCHS, epochs=EPOCHS, peak_lr=PEAK_LR):
    out = weights_path(round_name, fold)
    if os.path.exists(out):
        print(f"[{round_name} fold {fold}] already trained — skipping ({out})")
        return out

    keras.backend.clear_session()
    gc.collect()
    keras.utils.set_random_seed(SEED + fold)

    train = df[df.fold != fold]
    if extra is not None:
        train = pd.concat([train, extra], ignore_index=True)
    val = df[df.fold == fold]
    train_ds, val_ds = make_train_ds(train), make_eval_ds(val, labels=True)
    steps = len(train) // BATCH_SIZE
    print(f"[{round_name} fold {fold}] train {len(train)} | val {len(val)} | {steps} steps/epoch")

    model, base = build_model(weights=None if init_weights else IMAGENET_WEIGHTS)
    if init_weights:
        model.load_weights(init_weights)
    elif head_epochs:
        set_base_trainable(base, False)
        compile_model(model, 1e-3, steps * head_epochs, 0)
        model.fit(train_ds, validation_data=val_ds, epochs=head_epochs, verbose=2)

    set_base_trainable(base, True)
    compile_model(model, peak_lr, steps * epochs, warmup_steps=steps)
    hist = model.fit(train_ds, validation_data=val_ds, epochs=epochs, verbose=2)

    model.save_weights(out)
    pd.DataFrame(hist.history).to_csv(out.replace(".weights.h5", "_history.csv"), index=False)
    print(f"[{round_name} fold {fold}] final val_accuracy {hist.history['val_accuracy'][-1]:.4f}")
    return out


def predict_tta(model, frame):
    """Average of the image and its horizontal flip — the backend does the same."""
    ds = make_eval_ds(frame)
    p = model.predict(ds, verbose=0)
    p_flip = model.predict(ds.map(lambda x: tf.reverse(x, axis=[2])), verbose=0)
    return (p + p_flip) / 2


def predict_folds(weight_files):
    """Out-of-fold predictions for df, and the ensemble average for test_df."""
    oof = np.full((len(df), NUM_CLASSES), np.nan, dtype="float32")
    test_probs = np.zeros((len(test_df), NUM_CLASSES), dtype="float32")
    for fold, path in weight_files.items():
        keras.backend.clear_session()
        model, _ = build_model(weights=None)
        model.load_weights(path)
        val_mask = (df.fold == fold).values
        oof[val_mask] = predict_tta(model, df[val_mask])
        test_probs += predict_tta(model, test_df) / len(weight_files)
    return oof, test_probs


def oof_accuracy(oof):
    mask = ~np.isnan(oof[:, 0])
    return float((oof[mask].argmax(1) == df.y.values[mask]).mean())

## 7. Round 1 — train one model per fold

Roughly: EfficientNetV2-S takes 1–2 min per epoch on an A100 and 4–6 min on a T4.

In [ ]:
round1 = {fold: train_fold(fold, "round1") for fold in FOLDS_TO_TRAIN}
oof1, test_probs1 = predict_folds(round1)
print(f"Round 1 out-of-fold accuracy (single model + flip TTA): {oof_accuracy(oof1):.4f}")

## 8. Round 2 — pseudo-labelling

The Kaggle test images have no labels. The round-1 ensemble labels the ones it is ≥95% sure about;
each fold model is then fine-tuned a little more on its training data plus those images.
Validation folds never contain pseudo-labelled images, so the reported accuracy stays honest.

In [ ]:
if PSEUDO_LABEL:
    conf = test_probs1.max(1)
    keep = conf >= PSEUDO_THRESHOLD
    pseudo = test_df[keep].copy()
    pseudo["y"] = test_probs1[keep].argmax(1)
    pseudo["w"] = CLASS_WEIGHTS[pseudo.y]
    print(f"Pseudo-labelled {keep.sum()} of {len(test_df)} test images")
    print(pd.Series([CLASS_NAMES[i] for i in pseudo.y]).value_counts())

    final = {
        fold: train_fold(fold, "round2", extra=pseudo[["path", "y", "w"]], init_weights=round1[fold],
                         head_epochs=0, epochs=PL_EPOCHS, peak_lr=PL_PEAK_LR)
        for fold in FOLDS_TO_TRAIN
    }
    oof, _ = predict_folds(final)
    print(f"Round 2 out-of-fold accuracy: {oof_accuracy(oof):.4f}  (round 1: {oof_accuracy(oof1):.4f})")
else:
    final, oof = round1, oof1

## 9. Evaluation

Out-of-fold = every image is predicted by the one model that never trained on it. This is the honest
single-model number; the deployed 5-model ensemble is usually a little better.

In [ ]:
mask = ~np.isnan(oof[:, 0])
y_true, y_pred = df.y.values[mask], oof[mask].argmax(1)
pd.set_option("display.width", 250)
pd.set_option("display.max_columns", 20)
print(classification_report(y_true, y_pred, target_names=CLASS_NAMES, digits=3, zero_division=0))
print("Confusion matrix (rows = actual, cols = predicted):")
print(pd.DataFrame(confusion_matrix(y_true, y_pred), index=CLASS_NAMES, columns=[c[:8] for c in CLASS_NAMES]))

## 10. Confidence calibration and threshold

Temperature scaling makes "80% confident" mean right about 80% of the time. The threshold is then the
lowest confidence at which accepted predictions reach `TARGET_ACCEPTED_ACCURACY`; below it the app
tells the farmer the result is uncertain.

In [ ]:
def calibrate(probs, T):
    z = np.log(np.clip(probs, 1e-12, 1.0)) / T
    z -= z.max(1, keepdims=True)
    e = np.exp(z)
    return e / e.sum(1, keepdims=True)


def nll(T):
    return -np.log(np.clip(calibrate(oof[mask], T)[np.arange(mask.sum()), y_true], 1e-12, 1)).mean()


def ece(probs, y, bins=15):
    conf, pred = probs.max(1), probs.argmax(1)
    edges = np.linspace(0, 1, bins + 1)
    total = 0.0
    for lo, hi in zip(edges[:-1], edges[1:]):
        m = (conf > lo) & (conf <= hi)
        if m.any():
            total += m.mean() * abs((pred[m] == y[m]).mean() - conf[m].mean())
    return total


TEMPERATURE = float(minimize_scalar(nll, bounds=(0.1, 10.0), method="bounded").x)
cal = calibrate(oof[mask], TEMPERATURE)
print(f"Temperature {TEMPERATURE:.3f} | calibration error {ece(oof[mask], y_true):.4f} -> {ece(cal, y_true):.4f}")

conf, correct = cal.max(1), cal.argmax(1) == y_true
rows = []
for t in range(30, 100, 5):
    accepted = conf >= t / 100
    rows.append({"threshold %": t, "photos accepted": f"{accepted.mean():.1%}",
                 "accuracy on accepted": correct[accepted].mean() if accepted.any() else np.nan})
table = pd.DataFrame(rows)
print(table.to_string(index=False))

ok = table[table["accuracy on accepted"] >= TARGET_ACCEPTED_ACCURACY]
RECOMMENDED_THRESHOLD = float(ok["threshold %"].iloc[0]) if len(ok) else 90.0
print(f"Recommended CONFIDENCE_THRESHOLD: {RECOMMENDED_THRESHOLD:.0f}%")

## 11. Export to TFLite

One `.tflite` per fold plus `model_meta.json` (class order, input size, temperature, threshold).

In [ ]:
keras.mixed_precision.set_dtype_policy("float32")   # export a plain float32 graph
TFLITE_DIR = os.path.join(WORK_DIR, "tflite")
shutil.rmtree(TFLITE_DIR, ignore_errors=True)
os.makedirs(TFLITE_DIR)

model_files = []
for fold, path in final.items():
    keras.backend.clear_session()
    model, _ = build_model(weights=None)
    model.load_weights(path)
    export_dir = os.path.join(WORK_DIR, f"savedmodel_fold{fold}")
    model.export(export_dir)

    converter = tf.lite.TFLiteConverter.from_saved_model(export_dir)
    if QUANTIZE == "dynamic":
        converter.optimizations = [tf.lite.Optimize.DEFAULT]
    elif QUANTIZE == "float16":
        converter.optimizations = [tf.lite.Optimize.DEFAULT]
        converter.target_spec.supported_types = [tf.float16]
    name = f"paddy_disease_model_fold{fold}.tflite"
    with open(os.path.join(TFLITE_DIR, name), "wb") as f:
        f.write(converter.convert())
    model_files.append(name)
    print(f"{name}: {os.path.getsize(os.path.join(TFLITE_DIR, name)) / 1e6:.1f} MB")

meta = {
    "models": model_files,
    "class_names": CLASS_NAMES,
    "backbone": BACKBONE,
    "input_height": IMG_H,
    "input_width": IMG_W,
    "preprocess": "none",
    "tta": "hflip",
    "temperature": round(TEMPERATURE, 4),
    "recommended_threshold": RECOMMENDED_THRESHOLD,
    "oof_accuracy": round(oof_accuracy(oof), 4),
    "quantize": QUANTIZE,
    "pseudo_label": PSEUDO_LABEL,
    "trained_on": str(date.today()),
}
with open(os.path.join(TFLITE_DIR, "model_meta.json"), "w") as f:
    json.dump(meta, f, indent=2)
print(json.dumps(meta, indent=2))

## 12. Check the exported models the way the backend runs them

Loads original full-size photos with exactly the backend's preprocessing (PIL bilinear resize, flip TTA)
and compares TFLite with the Keras model. If quantization costs more than ~0.5%, set
`QUANTIZE = "float16"` and re-run section 11.

In [ ]:
def serve_input(src, w, h):
    """Mirror of backend/inference.py: prepare_image + PIL bilinear resize, raw 0-255 floats."""
    with Image.open(src) as img:
        img = prepare_image(img).resize((w, h), Image.Resampling.BILINEAR)
    return np.asarray(img, dtype=np.float32)


def tflite_tta(interpreter, arr):
    inp = interpreter.get_input_details()[0]["index"]
    out = interpreter.get_output_details()[0]["index"]
    probs = []
    for x in (arr, arr[:, ::-1, :]):
        interpreter.set_tensor(inp, np.ascontiguousarray(x[None]))
        interpreter.invoke()
        probs.append(interpreter.get_tensor(out)[0])
    return np.mean(probs, 0)


for fold, path in final.items():
    sample = df[df.fold == fold].sample(min(TFLITE_CHECK_SAMPLES, (df.fold == fold).sum()), random_state=SEED)
    arrs = np.stack([serve_input(s, IMG_W, IMG_H) for s in sample.src])

    keras.backend.clear_session()
    model, _ = build_model(weights=None)
    model.load_weights(path)
    p_keras = (model.predict(arrs, verbose=0) + model.predict(arrs[:, :, ::-1, :], verbose=0)) / 2

    interpreter = tf.lite.Interpreter(os.path.join(TFLITE_DIR, f"paddy_disease_model_fold{fold}.tflite"))
    interpreter.allocate_tensors()
    p_tfl = np.stack([tflite_tta(interpreter, a) for a in arrs])

    y = sample.y.values
    print(f"fold {fold}: keras {(p_keras.argmax(1) == y).mean():.4f} | tflite {(p_tfl.argmax(1) == y).mean():.4f}"
          f" | agreement {(p_keras.argmax(1) == p_tfl.argmax(1)).mean():.4f}")

## 13. Save and download

Copy everything in the downloaded `tflite` folder into the repo's `backend/models/`
(`paddy_disease_model_fold*.tflite` and `model_meta.json`). With `MODEL_PATH=backend/models`
(the default) the backend loads only the models listed in `model_meta.json`; the old v2 file is ignored.

In [ ]:
dest = os.path.join(SAVE_DIR, "tflite")
shutil.copytree(TFLITE_DIR, dest, dirs_exist_ok=True)
archive = shutil.make_archive(os.path.join(WORK_DIR, "kisan_mitra_v3_models"), "zip", TFLITE_DIR)
print("Saved to", dest)
print("Zip:", archive)

In [ ]:
from google.colab import files
files.download(archive)

## 14. Try your own photos

Runs the exported ensemble exactly as the app does. Upload real farmer photos here — this is the
best check of real-world accuracy.

In [ ]:
from google.colab import files
uploaded = files.upload()
interpreters = []
for name in model_files:
    it = tf.lite.Interpreter(os.path.join(TFLITE_DIR, name))
    it.allocate_tensors()
    interpreters.append(it)

for filename in uploaded:
    arr = serve_input(filename, IMG_W, IMG_H)
    probs = np.mean([tflite_tta(it, arr) for it in interpreters], 0)
    probs = calibrate(probs[None], TEMPERATURE)[0]
    top = probs.argsort()[::-1][:3]
    verdict = "OK" if probs[top[0]] * 100 >= RECOMMENDED_THRESHOLD else "LOW CONFIDENCE"
    print(f"{filename}: " + ", ".join(f"{CLASS_NAMES[i]} {probs[i]:.1%}" for i in top) + f"  [{verdict}]")